# Thai DSR — expanded-dataset L1 baseline (step 2)

A fresh bi-LSTM layer-9 mapper (L1 only, same W5 recipe: lr 1e-3, batch 4, reduce-on-plateau) trained on 2,010 severe utterances (10 own + 2,000 Common Voice Thai).
The split keeps every W5 assignment, so the W5 32 val / 8 STOI-PESQ utterances are unchanged: **train 1,407 / val 302 / test 301**.

1. Upload `colab_expanded_baseline_bundle.zip` to **My Drive/thai_dsr_colab/**.
2. **Runtime → Change runtime type → T4 GPU (or better)**, then run cells 1–7 in order.

Checkpoints, logs and the comparison go to **My Drive/thai_dsr_colab/expanded_baseline/**. After a disconnect: rerun cells 1–5 (extraction skips nothing locally — `/content` is wiped — but takes only a few minutes on GPU), then cell 6 resumes from the newest periodic checkpoint.

In [ ]:
# 1. GPU check
import torch
print('torch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime (Runtime -> Change runtime type), then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/thai_dsr_colab')   # EDIT if you used another folder
BUNDLE = DRIVE_ROOT / 'colab_expanded_baseline_bundle.zip'
RUN_DIR = DRIVE_ROOT / 'expanded_baseline'
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file(), f'Upload the zip to {BUNDLE}'
print('bundle bytes:', BUNDLE.stat().st_size)

In [ ]:
# 3. Unzip the bundle to /content/thai-dsr and verify every file (SHA-256)
import hashlib, json, os, shutil, subprocess, sys, zipfile
REPO = Path('/content/thai-dsr')
REPO.mkdir(exist_ok=True)
local_zip = Path('/content/bundle.zip')
shutil.copyfile(BUNDLE, local_zip)          # one sequential Drive read is much faster than 4k small ones
with zipfile.ZipFile(local_zip) as archive:
    for entry in archive.infolist():
        assert (REPO / entry.filename).resolve().is_relative_to(REPO.resolve()), entry.filename
    archive.extractall(REPO)
local_zip.unlink()
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for entry in manifest['files']:
    p = REPO / entry['path']
    assert p.stat().st_size == entry['bytes'], p
    digest = hashlib.sha256()
    with p.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    assert digest.hexdigest() == entry['sha256'], f'Checksum mismatch: {p}'
os.chdir(REPO)
print('Verified', len(manifest['files']), 'files in', REPO)

In [ ]:
# 4. Dependencies (keep Colab's CUDA torch/torchaudio; transformers pinned to the local environment)
import importlib.metadata as metadata
constraints = Path('/content/thai_dsr_constraints.txt')
constraints.write_text(''.join(f'{n}=={metadata.version(n)}\n' for n in ('torch', 'torchaudio')))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
                'PyYAML>=6.0', 'transformers==5.13.1', '-c', str(constraints)], check=True)
subprocess.run([sys.executable, '-c',
    'import torch, torchaudio, yaml, pesq, pystoi; from transformers import Wav2Vec2Model; '
    'assert torch.cuda.is_available(); print("ok", torch.__version__, torchaudio.__version__)'], check=True)

In [ ]:
# 5. wav2vec2 layer-9 embeddings on GPU for all 2,010 severe clips (~1.8 GB in /content)
# Recomputed for the old 210 as well, so baseline and new mapper are scored on identical features.
os.environ['HF_HOME'] = '/content/thai_dsr_hf'
os.environ.pop('HF_HUB_OFFLINE', None)
for src in ('data/distorted', 'data/distorted_cv', 'data/distorted_cv_ext'):
    subprocess.run(f'{sys.executable} -m src.preprocessing.extract_embedding --input_dir {src} '
                   f'--output_dir data/embeddings/distorted --layer 9 | grep -v "^Layer 09 shape" | tail -3',
                   shell=True, check=True)
emb = sorted((REPO / 'data/embeddings/distorted').glob('*_severe_layer09.npy'))
print('layer-9 embeddings:', len(emb))
assert len(emb) == 2010, 'expected 2010 cached embeddings'
from transformers import AutoConfig
rev = AutoConfig.from_pretrained('airesearch/wav2vec2-large-xlsr-53-th')._commit_hash
print('encoder revision:', rev, '(local runs used 3155938c549b23eee16b1d4b55dcb161b7fe4bcf)')

In [ ]:
# 6. Train (early stop: 6 epochs without val improvement, max 40 epochs). Resumes automatically.
import re
periodic = sorted(RUN_DIR.glob('mapper_layer9_expanded_epoch_*.pt'),
                  key=lambda p: int(re.search(r'epoch_(\d+)', p.name).group(1)))
RESUME = periodic[-1] if periodic else None   # or set an explicit Path
command = [sys.executable, '-u', '-m', 'src.training.train',
           '--train-config', 'configs/train_expanded.yaml',
           '--checkpoint-dir', str(RUN_DIR), '--log-dir', str(RUN_DIR / 'logs'),
           '--best-checkpoint', str(RUN_DIR / 'mapper_layer9_expanded.pt'),
           '--loss-curve', str(RUN_DIR / 'loss_curve_expanded.png'),
           '--early-stop-patience', '6', '--num-workers', '2']
if RESUME:
    command += ['--resume-checkpoint', str(RESUME)]
    print('Resuming from', RESUME)
print(' '.join(command), flush=True)
log_path = RUN_DIR / 'logs' / 'train.log'
log_path.parent.mkdir(exist_ok=True)
with log_path.open('a') as log:
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        if 'it/s' not in line and 's/it' not in line:   # drop tqdm batch bars
            print(line, end='')
assert proc.wait() == 0, f'training failed; see {log_path}'

In [ ]:
# 7. Compare with the W5 baseline (mapper_layer9.pt) using the exact W5 smoke-validation protocol
BASE = REPO / 'results/checkpoints/mapper_layer9.pt'
if not BASE.exists():   # taken from the earlier joint-finetune bundle already on Drive
    old = DRIVE_ROOT / 'colab_transfer_bundle.zip'
    assert old.is_file(), f'Put mapper_layer9.pt at {BASE} or keep {old} on Drive'
    with zipfile.ZipFile(old) as archive:
        archive.extract('results/checkpoints/mapper_layer9.pt', REPO)
out = RUN_DIR / 'comparison'
subprocess.run([sys.executable, 'scripts/evaluate_expanded_mapper.py',
                '--checkpoint', f'baseline={BASE.relative_to(REPO)}',
                '--checkpoint', f'expanded={RUN_DIR / "mapper_layer9_expanded.pt"}',
                '--output-dir', str(out)], check=True)
result = json.loads((out / 'comparison.json').read_text())
print(f"{'model':28s} {'W5 val L1':>9s} {'exp val L1':>10s} {'STOI(8)':>8s} {'PESQ(8)':>8s} {'STOI(24 new)':>12s} {'PESQ(24 new)':>12s}")
for name, m in result['models'].items():
    print(f"{name:28s} {m['w5_val_mel_l1']:9.4f} {m['expanded_val_mel_l1']:10.4f} {m['w5_audio']['stoi']:8.4f} "
          f"{m['w5_audio']['pesq']:8.4f} {m['extra_audio']['stoi']:12.4f} {m['extra_audio']['pesq']:12.4f}")
print('Reference (local W5 smoke baseline): val L1 1.2885, STOI 0.2615, PESQ 1.0944')